# bounds-gemma · Fine-tune Gemma 4 E2B on healthcare PHI

Trains a LoRA adapter on `gemma-4-E2B-it` using a synthetic clinical-note corpus (generated by `generate_phi_dataset.mjs`), merges it back into the base, exports to GGUF + LiteRT-LM, and (optionally) pushes to Hugging Face.

**Runtime:** Colab T4 (16 GB VRAM) is the target. Total wall-clock ~60-90 minutes for 500 examples × 3 epochs.

**Inputs you bring:**
- `phi_training.jsonl` from running `generate_phi_dataset.mjs` locally
- A Hugging Face token with write access to the `Aqta-ai/*` org

**Outputs:**
- LoRA adapter weights (~50 MB)
- Merged fp16 model (~5 GB)
- GGUF q4_k_m for Ollama (~3 GB)
- LiteRT-LM `.task` for WebLLM/MediaPipe (~1.5-2 GB)

## 1. Verify GPU

In [ ]:
!nvidia-smi | head -20

## 2. Install dependencies

In [ ]:
%%capture
!pip install --quiet unsloth datasets transformers trl peft accelerate bitsandbytes
!pip install --quiet ai-edge-torch ai-edge-quantizer

## 3. Clone the bounds-gemma repo (private during the submission)

If your Hugging Face / GitHub credentials are in Colab Secrets, the clone will work directly. Otherwise upload the repo as a zip and unzip into `/content/bounds-gemma`.

In [ ]:
import os
from google.colab import userdata

GH_TOKEN = userdata.get('GH_TOKEN')  # Set in Colab Secrets first
if GH_TOKEN:
    !git clone https://{GH_TOKEN}@github.com/Aqta-ai/bounds-gemma.git /content/bounds-gemma
else:
    print('No GH_TOKEN found. Upload the repo as a zip and extract to /content/bounds-gemma')

%cd /content/bounds-gemma

## 4. Upload the training data

Drag `finetune/data/phi_training.jsonl` (from your local machine, generated by `generate_phi_dataset.mjs`) into the Files panel on the left, into the `finetune/data/` directory of this Colab session.

In [ ]:
import os
assert os.path.exists('finetune/data/phi_training.jsonl'), 'Upload phi_training.jsonl before continuing'
!wc -l finetune/data/phi_training.jsonl

## 5. Train

In [ ]:
!python finetune/scripts/train.py

## 6. Smoke-test the fine-tune

Load the merged model and run an inference on a sample clinical note. Verify the model emits valid JSON with detections from the six PHI categories.

In [ ]:
from unsloth import FastLanguageModel
import json

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='bounds-gemma-e2b-phi-ft/merged',
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

SAMPLE = """Patient is a 42 year old who presents with generalised anxiety disorder. She has been on lithium for three years and has been taking Adderall for ADHD since her teens. She underwent CABG last April. Family history is significant for Huntington's disease on her father's side. She mentioned her therapist on Tuesday and her insulin pump alarm woke her at 3am."""

SYS = open('finetune/scripts/generate_phi_dataset.mjs').read()
# Pull PRODUCTION_SYSTEM_PROMPT from the script (between the triple-backticks). Or copy-paste from src/workers/gemma.worker.ts.
import re
m = re.search(r'PRODUCTION_SYSTEM_PROMPT = `([^`]+)`', SYS)
SYSTEM = m.group(1) if m else ''

messages = [
    {'role': 'system', 'content': SYSTEM},
    {'role': 'user', 'content': SAMPLE},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=1024, temperature=0.1, do_sample=True)
raw = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print('Raw model output:')
print(raw[:2000])
print()
try:
    detections = json.loads(raw)
    print(f'Parsed {len(detections)} detection(s)')
    for d in detections:
        print(f"  - [{d.get('ruleId', '?')}] (conf {d.get('confidence', '?')}): \"{d.get('text', '')[:60]}\"")
except json.JSONDecodeError as err:
    print(f'JSON parse failed: {err}')

## 7. Export to LiteRT-LM .task

In [ ]:
!python finetune/scripts/export_litert.py

## 8. Push to Hugging Face

Requires a HF_TOKEN with write access to `Aqta-ai/*`. Set it in Colab Secrets.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login

HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    print('Set HF_TOKEN in Colab Secrets (Lock icon in left sidebar) and re-run.')
else:
    login(token=HF_TOKEN)
    api = HfApi()

    REPO = 'Aqta-ai/bounds-gemma-e2b-phi-ft'
    api.create_repo(REPO, exist_ok=True, private=False)

    # Upload the merged HF transformers model
    api.upload_folder(
        folder_path='bounds-gemma-e2b-phi-ft/merged',
        repo_id=REPO,
        repo_type='model',
        commit_message='Initial upload: Gemma 4 E2B fine-tuned on bounds-gemma PHI corpus',
    )

    # Upload the LiteRT-LM .task variants alongside (browser path)
    api.upload_file(
        path_or_fileobj='bounds-gemma-e2b-phi-ft.task',
        path_in_repo='bounds-gemma-e2b-phi-ft.task',
        repo_id=REPO,
        repo_type='model',
    )
    api.upload_file(
        path_or_fileobj='bounds-gemma-e2b-phi-ft-web.task',
        path_in_repo='bounds-gemma-e2b-phi-ft-web.task',
        repo_id=REPO,
        repo_type='model',
    )

    # And a separate repo for the GGUF (Ollama users)
    GGUF_REPO = 'Aqta-ai/bounds-gemma-e2b-phi-ft-gguf'
    api.create_repo(GGUF_REPO, exist_ok=True, private=False)
    api.upload_folder(
        folder_path='bounds-gemma-e2b-phi-ft-gguf',
        repo_id=GGUF_REPO,
        repo_type='model',
    )

    print(f'Pushed to:')
    print(f'  https://huggingface.co/{REPO}')
    print(f'  https://huggingface.co/{GGUF_REPO}')